> **Notebook-first lesson.** Run, perturb, measure, explain. The activity cell turns the lesson's core idea into an executable experiment.

## Mathematical Framework

Math companions for this lesson:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Use the math to separate **measured association from assumptions, optimization behavior from generalization, and confidence scores from calibrated uncertainty**.

# Lesson 71: Self-supervised learning

Labeling is expensive. Self-supervised learning creates learning signals from the data itself.

## Examples
- predict masked tokens
- predict next token
- contrast augmented views
- reconstruct corrupted inputs

## Contrastive intuition
Represent two related views near each other in embedding space while separating unrelated samples.

## Why it matters
A representation can be pretrained on large unlabeled corpora and later adapted to smaller labeled tasks.

## Signal connection
Possible pretext tasks:
- reconstruct masked waveform segments
- identify whether two windows came from the same recording
- contrast augmented spectrogram views

## Exercise
Build a small contrastive or reconstruction pretraining experiment, then compare downstream classification with and without pretraining.


## Runnable activity
Execute this reduced-scale experiment, then change one assumption and compare the result.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
import torch
from torch import nn
torch.manual_seed(0)
X=torch.randn(512,20)
# Two noisy views of the same samples
v1=X+.1*torch.randn_like(X); v2=X+.1*torch.randn_like(X)
enc=nn.Sequential(nn.Linear(20,32),nn.ReLU(),nn.Linear(32,8))
opt=torch.optim.Adam(enc.parameters(),lr=.01)
for _ in range(120):
    z1=nn.functional.normalize(enc(v1),dim=1); z2=nn.functional.normalize(enc(v2),dim=1)
    positive=(z1*z2).sum(1)
    # Simple alignment + variance-preserving penalty for educational use
    loss=(1-positive).mean()+.1*((z1.std(0)-.2)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print("mean positive cosine similarity",float((nn.functional.normalize(enc(v1),dim=1)*nn.functional.normalize(enc(v2),dim=1)).sum(1).mean()))

## Final checkpoint
Add a Markdown cell with: **mechanism, evidence, limitation, and what you would test next.**